# [5] 지방선거 읍면동 단위 개표결과 통합 (6~8회)

사전투표 도입(2014) 이후 지방선거(6회 2014 / 7회 2018 / 8회 2022)만 대상.

## 투표수 정의 (팀 결정사항)
투표용지 부족은 물리적으로 인쇄된 용지가 **선거일 당일 투표소**에서 소진될 때 발생하는 문제.
사전투표(관내/관외)는 통합선거인명부 기반 현장 발급 방식이라 "미리 준비한 용지가 부족해지는" 문제 자체가 다름.
→ **분자(투표수) = 선거일투표수만 사용**, **분모(선거인수) = 읍면동 전체 선거인수(소계)** 사용.

## 원본 파일 구조 (읍면동별 블록 반복)
| 읍면동 | 구분 | 의미 |
|---|---|---|
| 합계 / 계 | (공란) | 구시군 전체 합계 |
| 거소투표 | (공란) | 구시군 단위로만 집계 |
| 관외사전투표 | (공란) | 구시군 단위로만 집계 |
| 동이름 | 소계 | 읍면동 전체 선거인수/투표수 (관내사전투표+선거일투표) |
| 동이름 | 관내사전투표 | 세부 항목 |
| 동이름 | 선거일투표 | **당일 실제 사용 투표수 (이 프로젝트의 핵심 분자)** |


In [1]:
import pandas as pd
import numpy as np

def clean_num(s):
    if pd.isna(s):
        return np.nan
    if isinstance(s, (int, float)):
        return s
    return pd.to_numeric(str(s).replace(",", "").strip(), errors="coerce")


In [2]:
FILES = {
    "20140604": {
        "path": "중앙선거관리위원회_제6회 전국동시지방선거 개표결과_20140604.xlsx",
        "election_name": "제6회 전국동시지방선거",
        "cols": {"시도": "시도명", "구시군": "구시군명", "읍면동": "읍면동명"},
    },
    "20180613": {
        "path": "중앙선거관리위원회_제7회 전국동시지방선거 개표결과_20180613.xlsx",
        "election_name": "제7회 전국동시지방선거",
        "cols": {"시도명": "시도명", "구시군명": "구시군명", "읍면동명": "읍면동명"},
    },
    "20220601": {
        "path": "중앙선거관리위원회_제8회 전국동시지방선거 개표결과_20220601.xlsx",
        "election_name": "제8회 전국동시지방선거",
        "cols": {"선거구명": "시도명", "구시군명": "구시군명", "읍면동명": "읍면동명"},
    },
}
KEY = ["시도명", "구시군명", "읍면동명"]


In [3]:
all_dfs = []

for sg_id, info in FILES.items():
    df = pd.read_excel(info["path"])
    df = df.rename(columns=info["cols"])

    # 7회는 선거구명(=시도명)이 별도 컬럼으로 존재
    if "선거구명" in df.columns and sg_id == "20180613":
        df["시도명"] = df["선거구명"]
    # 시도명 병합셀 forward-fill ("합계" 총계행은 제외)
    df["시도명"] = df["시도명"].where(df["시도명"] != "합계").ffill()

    df["선거인수"] = df["선거인수"].apply(clean_num)
    df["투표수"] = df["투표수"].apply(clean_num)

    # 읍면동 전체 선거인수 (소계 행) -> 용지 준비 기준의 분모
    total = df[df["구분"] == "소계"][KEY + ["선거인수"]].rename(columns={"선거인수": "선거인수_총"})

    # 선거일(당일) 실제 사용 투표수 -> 물리적 용지 부족 위험의 분자
    dayof = df[df["구분"] == "선거일투표"][KEY + ["투표수"]].rename(columns={"투표수": "선거일투표수"})

    dong = total.merge(dayof, on=KEY, how="inner")
    dong["선거ID"] = sg_id
    dong["선거명"] = info["election_name"]
    dong = dong.dropna(subset=["선거인수_총", "선거일투표수"])
    all_dfs.append(dong)
    print(f"{info['election_name']}: {len(dong)}개 읍면동")

merged = pd.concat(all_dfs, ignore_index=True)
merged = merged[["선거ID","선거명","시도명","구시군명","읍면동명","선거인수_총","선거일투표수"]]
merged.shape


제6회 전국동시지방선거: 3486개 읍면동


제7회 전국동시지방선거: 3496개 읍면동


제8회 전국동시지방선거: 3510개 읍면동


(10492, 7)

In [4]:
merged["당일투표율"] = merged["선거일투표수"] / merged["선거인수_총"]
merged["준비_50"] = merged["선거인수_총"] * 0.50
merged["준비_60"] = merged["선거인수_총"] * 0.60
merged["부족_50"] = (merged["선거일투표수"] - merged["준비_50"]).clip(lower=0)
merged["부족_60"] = (merged["선거일투표수"] - merged["준비_60"]).clip(lower=0)
merged.head()


,선거ID,선거명,시도명,구시군명,읍면동명,선거인수_총,선거일투표수,당일투표율,준비_50,준비_60,부족_50,부족_60
0,20140604,제6회 전국동시지방선거,서울특별시,종로구,청운효자동,11003.0,5933.0,0.539217,5501.5,6601.8,431.5,0.0
1,20140604,제6회 전국동시지방선거,서울특별시,종로구,사직동,8044.0,4060.0,0.504724,4022.0,4826.4,38.0,0.0
2,20140604,제6회 전국동시지방선거,서울특별시,종로구,삼청동,2582.0,1292.0,0.500387,1291.0,1549.2,1.0,0.0
3,20140604,제6회 전국동시지방선거,서울특별시,종로구,부암동,8536.0,4462.0,0.522727,4268.0,5121.6,194.0,0.0
4,20140604,제6회 전국동시지방선거,서울특별시,종로구,평창동,15493.0,8012.0,0.517137,7746.5,9295.8,265.5,0.0


In [5]:
# 선거별 전국 당일투표율 + 50/60% 기준 부족 발생 동 개수 확인
summary = merged.groupby("선거명").agg(
    선거인수_총=("선거인수_총","sum"),
    선거일투표수=("선거일투표수","sum"),
    부족동_50=("부족_50", lambda s: (s>0).sum()),
    부족동_60=("부족_60", lambda s: (s>0).sum()),
    총동수=("읍면동명","count"),
)
summary["당일투표율_전국"] = (summary["선거일투표수"]/summary["선거인수_총"]*100).round(2)
summary


,선거인수_총,선거일투표수,부족동_50,부족동_60,총동수,당일투표율_전국
선거명,,,,,,
제6회 전국동시지방선거,39098536.0,18600094.0,1567,318,3486,47.57
제7회 전국동시지방선거,39918492.0,17110704.0,325,6,3496,42.86
제8회 전국동시지방선거,41622478.0,13350334.0,5,0,3510,32.07


In [6]:
merged.to_csv("05_지방선거_읍면동_통합_6to8회.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", merged.shape)


저장 완료: (10492, 12)


## 분모 교체: 개표결과 소계 선거인수 → API 확정선거인수(계)

개표결과 파일의 읍면동 "소계" 선거인수는 관외사전투표·거소투표 등록자를 제외하고 있어 과소산정됨(평균 약 6.5% 낮음).
API(선거통계시스템 ElcntInfoInqireService)의 확정선거인수(계)가 실제 공식 선거인명부 등재자 수(그 동네 전체 유권자, 사전투표/거소투표 포함)이므로 이걸 분모로 사용.


In [7]:
api = pd.read_csv("02_선거인수정보_읍면동별.csv")
dong_api = api[api["읍면동명"] != "합계"][["선거명","시도명","구시군명","읍면동명","확정선거인수(계)"]]

merged = merged.merge(dong_api, on=["선거명","시도명","구시군명","읍면동명"], how="left")
print("매칭 안 된 행:", merged["확정선거인수(계)"].isna().sum())

merged = merged.rename(columns={"선거인수_총": "선거인수_개표결과기준"})
merged["선거인수_총"] = merged["확정선거인수(계)"]

# 최종 계산 재수행
merged["당일투표율"] = merged["선거일투표수"] / merged["선거인수_총"]
merged["준비_50"] = merged["선거인수_총"] * 0.50
merged["준비_60"] = merged["선거인수_총"] * 0.60
merged["부족_50"] = (merged["선거일투표수"] - merged["준비_50"]).clip(lower=0)
merged["부족_60"] = (merged["선거일투표수"] - merged["준비_60"]).clip(lower=0)

merged = merged[["선거ID","선거명","시도명","구시군명","읍면동명",
                  "선거인수_총","선거인수_개표결과기준","선거일투표수",
                  "당일투표율","준비_50","준비_60","부족_50","부족_60"]]
merged.head()


매칭 안 된 행: 0


,선거ID,선거명,시도명,구시군명,읍면동명,선거인수_총,선거인수_개표결과기준,선거일투표수,당일투표율,준비_50,준비_60,부족_50,부족_60
0,20140604,제6회 전국동시지방선거,서울특별시,종로구,청운효자동,11768,11003.0,5933.0,0.504164,5884.0,7060.8,49.0,0.0
1,20140604,제6회 전국동시지방선거,서울특별시,종로구,사직동,8552,8044.0,4060.0,0.474743,4276.0,5131.2,0.0,0.0
2,20140604,제6회 전국동시지방선거,서울특별시,종로구,삼청동,2772,2582.0,1292.0,0.466089,1386.0,1663.2,0.0,0.0
3,20140604,제6회 전국동시지방선거,서울특별시,종로구,부암동,9172,8536.0,4462.0,0.486481,4586.0,5503.2,0.0,0.0
4,20140604,제6회 전국동시지방선거,서울특별시,종로구,평창동,16418,15493.0,8012.0,0.488001,8209.0,9850.8,0.0,0.0


In [8]:
summary = merged.groupby("선거명").agg(
    선거인수_총=("선거인수_총","sum"),
    선거일투표수=("선거일투표수","sum"),
    부족동_50=("부족_50", lambda s: (s>0).sum()),
    부족동_60=("부족_60", lambda s: (s>0).sum()),
    총동수=("읍면동명","count"),
)
summary["당일투표율_전국"] = (summary["선거일투표수"]/summary["선거인수_총"]*100).round(2)
summary


,선거인수_총,선거일투표수,부족동_50,부족동_60,총동수,당일투표율_전국
선거명,,,,,,
제6회 전국동시지방선거,41296228,18600094.0,950,74,3486,45.04
제7회 전국동시지방선거,42907715,17110704.0,43,0,3496,39.88
제8회 전국동시지방선거,44303449,13350334.0,0,0,3510,30.13


In [9]:
merged.to_csv("05_지방선거_읍면동_통합_6to8회.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", merged.shape)


저장 완료: (10492, 13)


## Streamlit 대시보드용 추가 컴럼

- 관내사전투표비중: 레이더 차트의 한 축(사전투표 선호도)로 사용
- 각 지표의 선거별 백분위: 같은 선거 내 다른 동들과의 상대 위치를 0~100으로 표현 (레이더 차트 축 정규화용)
- 당일투표율_변동성: 한 동이 6~8회에 걸쳐 당일투표율이 얼마나 들쓑날드리었는지(표준편차). 행정구역 개편으로 3개 선거 데이터가 다 있는 동이 아닌 경우(295개 동은 1개 선거에만 존재) 변동성을 계산할 수 없어 NaN으로 남음.

In [10]:
# 관내사전투표 비중 (레이더 차트 축: 사전투표 선호도)
KEY = ["시도명","구시군명","읍면동명"]
all_insa = []
for sg_id, info in FILES.items():
    raw = pd.read_excel(info["path"])
    raw = raw.rename(columns=info["cols"])
    if "선거구명" in raw.columns and sg_id == "20180613":
        raw["시도명"] = raw["선거구명"]
    raw["시도명"] = raw["시도명"].where(raw["시도명"] != "합계").ffill()
    raw["투표수"] = raw["투표수"].apply(clean_num)

    insa = raw[raw["구분"] == "관내사전투표"][KEY + ["투표수"]].rename(columns={"투표수": "관내사전투표수"})
    insa["선거ID"] = sg_id; insa["선거명"] = info["election_name"]
    all_insa.append(insa)

insa_all = pd.concat(all_insa, ignore_index=True)
merged = merged.merge(insa_all, on=["선거ID","선거명"] + KEY, how="left")
merged["관내사전투표비중"] = merged["관내사전투표수"] / (merged["관내사전투표수"] + merged["선거일투표수"])
merged.groupby("선거명")["관내사전투표비중"].mean()


선거명
제6회 전국동시지방선거    0.139872
제7회 전국동시지방선거    0.284963
제8회 전국동시지방선거    0.381580
Name: 관내사전투표비중, dtype: float64

In [11]:
# 선거별 백분위 + 동별 6~8회 걸친 당일투표율 변동성
for col in ["당일투표율","부족_50","선거인수_총","관내사전투표비중"]:
    merged[col+"_pct"] = merged.groupby("선거명")[col].rank(pct=True) * 100

vol = merged.groupby(KEY)["당일투표율"].std().reset_index().rename(columns={"당일투표율":"당일투표율_변동성"})
vol["당일투표율_변동성_pct"] = vol["당일투표율_변동성"].rank(pct=True) * 100
merged = merged.merge(vol, on=KEY, how="left")

merged.to_csv("05_지방선거_읍면동_통합_6to8회.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", merged.shape)


저장 완료: (10492, 21)


## 절대량위험 / 비율위험 재정의 (팀 결정사항 v2)

기존에 대시보드에서 쓰던 "절대량위험(부족_50, clip 0)"과 "비율위험(당일투표율)"을 아래 정의로 교체.

**Risk_index_50** (읍면동 단위) = (선거일투표수 − 준비_50) / 선거인수_총 = 당일투표율 − 0.5
→ 양수 = 부족, 음수 = 여유. 기존 부족_50과 달리 **부호를 유지**(clip 안 함).

**절대량위험** = 선거일투표수 − 준비_50 (매수 단위, 부호 유지)
→ 읍면동 단위로 계산 후, 구시군/시도 단위는 하위 단위 값을 그대로 합산하면 동일하게 나옴
   (∵ Σ(투표수ᵢ − 0.5·선거인수ᵢ) = Σ투표수ᵢ − 0.5·Σ선거인수ᵢ). 대시보드에서 groupby-sum으로 집계.

**비율위험** = 그 지역 하위단위 Risk_index_50의 표준편차 ÷ 전국 평균 표준편차 (같은 선거 내)
→ 구시군 단위: 그 구 안의 동들의 Risk_index_50 표준편차 vs 전국 모든 구의 표준편차 평균
→ 시도 단위: 그 시도 안의 구들의 Risk_index_50(구 단위) 표준편차 vs 전국 모든 시도의 표준편차 평균
→ 값이 1보다 크면 "같은 평균 위험이라도 동네 간 변동성(격차)이 전국 평균보다 크다" = 어떤 동은 안전해 보여도 특정 동에서 몰아서 부족 사태가 날 위험이 큼.
→ 동(읍면동) 단위에서는 하위 단위가 없어 정의 불가 → 구/시도만 계산, 동 행에는 자신이 속한 구/시도 값을 참조용으로 붙임.


In [12]:
KEY = ["시도명","구시군명","읍면동명"]

# ---- 읍면동(동) 단위 ----
merged["Risk_index_50"] = merged["당일투표율"] - 0.5
merged["절대량위험"] = merged["선거일투표수"] - merged["준비_50"]

# ---- 구시군 단위 집계 ----
gu_risk = merged.groupby(["선거명","시도명","구시군명"]).agg(
    절대량위험_구=("절대량위험","sum"),
    표준편차_구=("Risk_index_50","std"),
).reset_index()
nat_avg_gu = gu_risk.groupby("선거명")["표준편차_구"].mean().rename("전국평균표준편차_구")
gu_risk = gu_risk.merge(nat_avg_gu, on="선거명", how="left")
gu_risk["비율위험_구"] = gu_risk["표준편차_구"] / gu_risk["전국평균표준편차_구"]

# ---- 시도 단위 집계 (구 단위 Risk_index_50을 하위 단위로 사용) ----
gu_risk["Risk_index_50_구"] = gu_risk["절대량위험_구"] / merged.groupby(["선거명","시도명","구시군명"])["선거인수_총"].sum().values

sido_risk = merged.groupby(["선거명","시도명"]).agg(
    절대량위험_시도=("절대량위험","sum"),
).reset_index()
sido_std = gu_risk.groupby(["선거명","시도명"])["Risk_index_50_구"].std().reset_index().rename(columns={"Risk_index_50_구":"표준편차_시도"})
sido_risk = sido_risk.merge(sido_std, on=["선거명","시도명"], how="left")
nat_avg_sido = sido_risk.groupby("선거명")["표준편차_시도"].mean().rename("전국평균표준편차_시도")
sido_risk = sido_risk.merge(nat_avg_sido, on="선거명", how="left")
sido_risk["비율위험_시도"] = sido_risk["표준편차_시도"] / sido_risk["전국평균표준편차_시도"]

# ---- 동 단위 CSV에 "자신이 속한 구/시도" 값 참조용으로 병합 ----
merged = merged.merge(
    gu_risk[["선거명","시도명","구시군명","절대량위험_구","표준편차_구","비율위험_구"]],
    on=["선거명","시도명","구시군명"], how="left"
)
merged = merged.merge(
    sido_risk[["선거명","시도명","절대량위험_시도","표준편차_시도","비율위험_시도"]],
    on=["선거명","시도명"], how="left"
)

print("구 단위 표본:")
print(gu_risk.head(3)[["선거명","시도명","구시군명","절대량위험_구","비율위험_구"]])
print()
print("시도 단위 표본:")
print(sido_risk.head(3)[["선거명","시도명","절대량위험_시도","비율위험_시도"]])
print()
print("비율위험_구 결측(동이 1개뿐인 구):", gu_risk["표준편차_구"].isna().sum(), "/", len(gu_risk))
print("비율위험_시도 결측(구가 1개뿐인 시도):", sido_risk["표준편차_시도"].isna().sum(), "/", len(sido_risk))


구 단위 표본:
            선거명  시도명 구시군명  절대량위험_구    비율위험_구
0  제6회 전국동시지방선거  강원도  강릉시  -7422.0  0.824989
1  제6회 전국동시지방선거  강원도  고성군   1229.5  1.664949
2  제6회 전국동시지방선거  강원도  동해시  -3823.0  0.640897

시도 단위 표본:
            선거명   시도명  절대량위험_시도   비율위험_시도
0  제6회 전국동시지방선거   강원도  -31795.5  1.221348
1  제6회 전국동시지방선거   경기도 -701117.5  1.143588
2  제6회 전국동시지방선거  경상남도  -69271.5  1.243324

비율위험_구 결측(동이 1개뿐인 구): 0 / 751
비율위험_시도 결측(구가 1개뿐인 시도): 3 / 51


In [13]:
merged.to_csv("05_지방선거_읍면동_통합_6to8회.csv", index=False, encoding="utf-8-sig")

# 구/시도 단위 결과도 별도 저장 (대시보드에서 map/aggregate용으로 바로 읽을 수 있게)
gu_risk.to_csv("05b_구단위_위험지표.csv", index=False, encoding="utf-8-sig")
sido_risk.to_csv("05c_시도단위_위험지표.csv", index=False, encoding="utf-8-sig")

print("저장 완료:", merged.shape, gu_risk.shape, sido_risk.shape)


저장 완료: (10492, 29) (751, 8) (51, 6)


In [14]:
# 레이더 차트 등 UI용 백분위 컬럼 (0~100, 같은 선거 내 상대 위치)
merged["절대량위험_pct"] = merged.groupby("선거명")["절대량위험"].rank(pct=True) * 100

gu_risk["비율위험_구_pct"] = gu_risk.groupby("선거명")["비율위험_구"].rank(pct=True) * 100
merged = merged.merge(
    gu_risk[["선거명","시도명","구시군명","비율위험_구_pct"]],
    on=["선거명","시도명","구시군명"], how="left"
)

merged.to_csv("05_지방선거_읍면동_통합_6to8회.csv", index=False, encoding="utf-8-sig")
gu_risk.to_csv("05b_구단위_위험지표.csv", index=False, encoding="utf-8-sig")
sido_risk.to_csv("05c_시도단위_위험지표.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", merged.shape)
print(merged[["선거명","시도명","구시군명","읍면동명","절대량위험_pct","비율위험_구_pct"]].head())


저장 완료: (10492, 31)
            선거명    시도명 구시군명   읍면동명  절대량위험_pct  비율위험_구_pct
0  제6회 전국동시지방선거  서울특별시  종로구  청운효자동  78.628801   53.784861
1  제6회 전국동시지방선거  서울특별시  종로구    사직동  55.421687   53.784861
2  제6회 전국동시지방선거  서울특별시  종로구    삼청동  63.998853   53.784861
3  제6회 전국동시지방선거  서울특별시  종로구    부암동  61.531842   53.784861
4  제6회 전국동시지방선거  서울특별시  종로구    평창동  56.927711   53.784861
